# ASE NEB relaxation script
This is a script to run a neb relaxation to find a transition state. This script can be run with minimal editing or be used as a base to develop you own scripts.

## Imports
These are the modules and functions that are needed for the script to work as intended. Additional imports can be added here if the script is modified. Remember to rerun this cell if any imports is added.

In [ ]:
from ase import io
from ase.optimize import FIRE, BFGS, MDMin, CellAwareBFGS
from ase.optimize.sciopt import SciPyFminCG

import pfp_api_client
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

## Matlantis calculator
Set the calculator that will be used to get the energy and forces for the relaxation. __Edit__ this cell to change the calculator mode (e.g. _PBE_, _PBE_PLUS_D3_, _R2SCAN_, _R2SCAN_PLUS_D3_). This will use the different versions of the potential fitted agains different DFT functionals.

In [ ]:
estimator = Estimator(calc_mode=EstimatorCalcMode.PBE_PLUS_D3, model_version='v9.0.0')
calculator = ASECalculator(estimator)

print(f'pfp_api_client: {pfp_api_client.__version__}')

## Parameters
Set these parameters to control the job (like a vasp INCAR). File paths should be absolute or relative to the directory where the notebook file is before job submission. There are two ways to run this script. If __path_file__ is set, the 2nd method will be used.
1) By giving the two endpoints and the number of images in the path. The path will be constructed by interpolation between the endpoints.
* __first_file__ is the first endpoint of the path to be relaxed. Will use the last frame if file contains multiple frames, use ase @ syntax to select a specific frame.
* __last_file__ is the last endpoint of the path to be relaxed. Will use the last frame if file contains multiple frames, use ase @ syntax to select a specific frame.
* __number_images__ is the number of images to create along the path (including the two endpoints).
2) By giving an already generated path.
* __path_file__ is the file for the path to be relaxed. Either set the images used in the cell or use the ase @ syntax.

Common parameters for both methods.
* __output_file__ is where to save the relaxed path. This file is only created when the relaxation finishes.
* __trajectory_file__ is where to save the path trajectory while relaxing. Use None to skip.
* __log_file__ is where to save the logging output for the relaxation. Use '-' to print to stdout (i.e. the notebook) and None to skip.
* __optimiser__ is the optimiser to use (i.e. IBRION or IOPT). Allowed options are FIRE, BFGS, MDMin, CellAwareBFGS and SciPyFminCG. My sugesstions are BFGS/SciPyFminCG when close to a minima or FIRE/MDMin when far from it.
* __fmax__ set the force convergence for the relaxation in eV/Å (i.e EDIFFG)
* __climb__ selects if climbing image neb should be used.

If the script is run in interactive mode or by the _Run in Background_ button edit these parameters directly in this cell. If the script is run by _mtl-bg-job_ the parameters can be overwriten by adding _-p parameter-name parameter-value_ to the command.

In [ ]:
first_file = 'first.traj'
last_file = 'last.traj'
number_images = 3

path_file = None

output_file = 'relaxed_neb.traj'
trajectory_file = None
log_file = '-'
optimiser = FIRE
fmax = 0.01
climb = False

Parameters read from _mtl-bg-job_ might have the wrong type. This cell will parse them correctly.

In [ ]:
if isinstance(optimiser, str):
    optimiser = {'FIRE': FIRE, 'BFGS':BFGS, 'MDMin':MDMin, 'CellAwareBFGS':CellAwareBFGS, 'SciPyFminCG':SciPyFminCG}[optimiser]

if path_file is None:
    number_images = int(number_images)

fmax = float(fmax)

if isinstance(climb, str):
    climb = (climb == 'True')

## Run NEB calculation
If the __path_file__ is set import the path. Otherwise, read the two endpoints and interpolate a path given the number of images. After the reading or constructing the path it will be relaxed and saved to the __output_file__.

In [ ]:
if path_file is None:
    images = io.read(path_file)
    neb = NEB(images)
    print('NEB path with {len(neb)} images loaded from {path_file}')
else:
    first_atoms = io.read(first_file)
    last_atoms = io.read(last_file)
    images = [first_atoms]
    images += [first_atoms.copy() for i in range(number_images - 2)]
    images += [last_atoms]

    neb = NEB(images)
    neb.interpolate()
    print('NEB path with {len(neb)} images created from {first_file} and {last_file}')

In [ ]:
for atoms in images:
    atoms.calc = ASECalculator(estimator)

dyn = optimiser(neb,
                logfile=log_file, loginterval=1)

traj = io.Trajectory(trajectory_file, 'w', atoms)
dyn.attach(traj)

try:
    status = dyn.run(fmax=fmax)
except:
    status = False
finally:
    io.write(output_file, atoms)
    print('Optimisation ended with status {status}.')